# Task 2: Naive RAG vs Contextual Retrieval

In this notebook, we:
1. Reload the extracted data.
2. Prompt for your Gemini API key dynamically.
3. Set up **Naive RAG**.
4. Set up **Contextual Retrieval** by enriching chunks with LLM-generated context.
5. Evaluate both pipelines using ROUGE on the generated 20 QA dataset.
6. Visualize the comparison.

In [ ]:
!pip install pymupdf faiss-cpu langchain langchain-community langchain-huggingface langchain-google-genai rouge-score matplotlib seaborn sentence-transformers

In [ ]:
import os
import json
import getpass
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import asyncio
import re
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from rouge_score import rouge_scorer

## 1. Environment & API Key
Please enter your **Gemini API Key** (using gemini-3.0-flash).

In [ ]:
api_key = getpass.getpass("Please enter your Gemini API Key: ")
os.environ["GEMINI_API_KEY"] = api_key

llm = ChatGoogleGenerativeAI(model="gemini-3.0-flash", temperature=0)
print("LLM initialized.")

## 2. Load Data and Base Vectors
We reload the PDF and recreate the base chunks. Then we use an open-source huggingface embedding model (`BAAI/bge-small-en-v1.5`) matching standard RAG practices to store embeddings.

In [ ]:
def clean_text(text):
    text = re.sub(r'\n+', '\n', text)
    text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'CHAPTER \d+.*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'Speech and Language Processing.*$', '', text, flags=re.MULTILINE)
    return text.strip()

pdf_path = "10.pdf"
loader = PyMuPDFLoader(pdf_path)
documents = loader.load()

full_text = ""
for doc in documents:
    full_text += clean_text(doc.page_content) + "\n"

# Create base chunks (Naive)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=100)
naive_chunks = text_splitter.create_documents([full_text])
print(f"Loaded {len(naive_chunks)} naive chunks.")

# Embeddings
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

# Naive Vectorstore
naive_vectorstore = FAISS.from_documents(naive_chunks, embeddings)
naive_retriever = naive_vectorstore.as_retriever(search_kwargs={"k": 3})
print("Naive RAG retriever ready.")

## 3. Contextual Retrieval
We enrich each chunk with brief context by prompting our LLM, and store these enriched chunks in a brand new vectorstore.

In [ ]:
async def enrich_chunk(chunk: str, document: str, title: str) -> str:
    """Add contextual prefix using LLM"""
    prompt = f"""
    Title: {title}
    {document[:4000]}
    {chunk}

    Provide brief context (1-2 sentences) explaining what this chunk discusses 
    in relation to the full document. Format: "This chunk from [title] discusses [explanation]."
    """
    response = await llm.ainvoke(prompt)
    context = response.content.strip()
    return f"{context}\n\n{chunk}"

import copy
contextual_chunks = copy.deepcopy(naive_chunks)

async def enrich_all():
    title = "Chapter 10: Transformers and Pretrained Language Models (Jurafsky & Martin)"
    tasks = []
    for c in contextual_chunks:
        tasks.append(enrich_chunk(c.page_content, full_text, title))
        
    # Run all tasks concurrently. Warning: May hit API rate limits if too fast.
    # To avoid rate limits, we process in batches.
    print("Starting chunk enrichment (this might take a minute)... ")
    batch_size = 10 
    enriched_contents = []
    for i in range(0, len(tasks), batch_size):
        batch = tasks[i:i+batch_size]
        results = await asyncio.gather(*batch)
        enriched_contents.extend(results)
        print(f"Processed {min(i+batch_size, len(tasks))} / {len(tasks)}")
        await asyncio.sleep(2) # rate limit buffer
        
    for i, content in enumerate(enriched_contents):
        contextual_chunks[i].page_content = content

# Await enrichment
await enrich_all()

# Create Contextual Vectorstore
contextual_vectorstore = FAISS.from_documents(contextual_chunks, embeddings)
contextual_vectorstore.save_local("answer/contextual_faiss_index")
naive_vectorstore.save_local("answer/naive_faiss_index")

contextual_retriever = contextual_vectorstore.as_retriever(search_kwargs={"k": 3})
print("Contextual RAG retriever ready.")

## 4. RAG Generation Functions
Given retrieved context, prompt the LLM to generate an answer.

In [ ]:
qa_prompt = PromptTemplate.from_template(
    "You are an expert NLP assistant.\n"
    "Answer the user's question clearly and precisely using ONLY the provided context.\n"
    "If the context does not contain the answer, say 'I don't know'.\n\n"
    "Context:\n{context}\n\n"
    "Question: {question}\n\n"
    "Answer:"
)

generator_chain = qa_prompt | llm

def generate_answer(question: str, retriever):
    docs = retriever.invoke(question)
    context = "\n---\n".join([d.page_content for d in docs])
    response = generator_chain.invoke({"context": context, "question": question})
    return response.content.strip()

## 5. Evaluation & Saving Results
We'll compute ROUGE over our 20 QA dataset.

In [ ]:
dataset_path = "answer/response-st-126010-chapter-10.json"
with open(dataset_path, "r") as f:
    qa_pairs = json.load(f)

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
naive_scores = {'r1': [], 'r2': [], 'rl': []}
contextual_scores = {'r1': [], 'r2': [], 'rl': []}

print("Evaluating Naive vs Contextual RAG...")
for i, item in enumerate(qa_pairs):
    q = item["question"]
    gt = item["ground_truth_answer"]
    
    # Generate
    ans_naive = generate_answer(q, naive_retriever)
    ans_contx = generate_answer(q, contextual_retriever)
    
    item["naive_rag_answer"] = ans_naive
    item["contextual_retrieval_answer"] = ans_contx
    
    # Score Naive
    scores_n = scorer.score(gt, ans_naive)
    naive_scores['r1'].append(scores_n['rouge1'].fmeasure)
    naive_scores['r2'].append(scores_n['rouge2'].fmeasure)
    naive_scores['rl'].append(scores_n['rougeL'].fmeasure)
    
    # Score Contextual
    scores_c = scorer.score(gt, ans_contx)
    contextual_scores['r1'].append(scores_c['rouge1'].fmeasure)
    contextual_scores['r2'].append(scores_c['rouge2'].fmeasure)
    contextual_scores['rl'].append(scores_c['rougeL'].fmeasure)

# Update JSON file with the generated answers
with open(dataset_path, "w") as f:
    json.dump(qa_pairs, f, indent=2)
    
print("Evaluations complete and saved.")

In [ ]:
# Averages
avg_n = {k: np.mean(v) for k, v in naive_scores.items()}
avg_c = {k: np.mean(v) for k, v in contextual_scores.items()}

import pandas as pd
results_df = pd.DataFrame([
    {"Method": "Naive RAG", "ROUGE-1": avg_n['r1'], "ROUGE-2": avg_n['r2'], "ROUGE-L": avg_n['rl']},
    {"Method": "Contextual Retrieval", "ROUGE-1": avg_c['r1'], "ROUGE-2": avg_c['r2'], "ROUGE-L": avg_c['rl']}
])

print(results_df.to_markdown(index=False))

## 6. Visualization

In [ ]:
sns.set_theme(style="whitegrid")

# Prepare data for plotting
melted_df = results_df.melt(id_vars="Method", var_name="Metric", value_name="Score")

plt.figure(figsize=(10, 6))
g = sns.barplot(data=melted_df, x="Metric", y="Score", hue="Method", palette="viridis")
plt.title("ROUGE Scores: Naive RAG vs Contextual Retrieval", fontsize=16)
plt.ylabel("F1 Score", fontsize=12)
plt.xlabel("Metric", fontsize=12)
plt.ylim(0, 1.0)
plt.legend(title="Retrieval Method")

for p in g.patches:
    g.annotate(format(p.get_height(), '.3f'), 
               (p.get_x() + p.get_width() / 2., p.get_height()), 
               ha = 'center', va = 'center', 
               xytext = (0, 9), 
               textcoords = 'offset points')

plt.savefig("answer/rouge_comparison.png", dpi=300)
plt.show()